## ML_1019: Causal Self-Attention

**Difficulty**: Hard | **TorchCode**: #09

### Problem
Implement autoregressive (causal) attention where position `i` can only attend to positions `j ≤ i`. Mask future positions with `-inf` before softmax.

### Formula
$$\text{scores}_{ij} = \frac{Q_i K_j^T}{\sqrt{d_k}} + M_{ij}, \quad M_{ij} = \begin{cases} 0 & j \le i \\ -\infty & j > i \end{cases}$$

In [1]:
import torch
import torch.nn as nn
import math


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, S, _ = x.shape

        q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)

        mask = torch.tril(torch.ones(S, S, device=x.device, dtype=torch.bool)).unsqueeze(0).unsqueeze(0)
        scores = scores.masked_fill(~mask, float("-inf"))

        max_value = scores.max(dim=-1, keepdim=True).values
        scores = scores - max_value
        e_x = torch.exp(scores)
        softmax = e_x / e_x.sum(dim=-1, keepdim=True)

        weight_value = softmax @ v

        output = weight_value.transpose(1, 2).contiguous().view(B, S, self.num_heads * self.d_k)
        output = self.W_o(output)
        return output

# --- Test 1: Output shape ---
torch.manual_seed(0)
csa = CausalSelfAttention(d_model=32, num_heads=4)
x = torch.randn(2, 6, 32)
out = csa.forward(x)
assert out.shape == (2, 6, 32)

# --- Test 2: nn.Linear projections with correct shapes ---
for name in ['W_q', 'W_k', 'W_v', 'W_o']:
    layer = getattr(csa, name)
    assert isinstance(layer, nn.Linear)
    assert layer.weight.shape == (32, 32)
    assert layer.weight.requires_grad

# --- Test 3: Numerical correctness vs reference ---
torch.manual_seed(0)
D, H = 16, 2
d_k = D // H
csa2 = CausalSelfAttention(d_model=D, num_heads=H)

x = torch.randn(1, 4, D)
out = csa2.forward(x)

q = csa2.W_q(x).view(1, 4, H, d_k).transpose(1, 2)
k = csa2.W_k(x).view(1, 4, H, d_k).transpose(1, 2)
v = csa2.W_v(x).view(1, 4, H, d_k).transpose(1, 2)

scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

mask = torch.tril(torch.ones(4, 4, device=x.device, dtype=torch.bool)).unsqueeze(0).unsqueeze(0)
scores = scores.masked_fill(~mask, float('-inf'))

ref = csa2.W_o(
    torch.matmul(torch.softmax(scores, dim=-1), v)
    .transpose(1, 2)
    .contiguous()
    .view(1, 4, D)
)

assert torch.allclose(out, ref, atol=1e-5)

# --- Test 4: Gradient flow ---
x = torch.randn(1, 4, 16, requires_grad=True)
csa2.forward(x).sum().backward()
assert x.grad is not None

# --- Test 5: Causal masking matrix check ---
torch.manual_seed(0)
x = torch.randn(1, 5, D)

q = csa2.W_q(x).view(1, 5, H, d_k).transpose(1, 2)
k = csa2.W_k(x).view(1, 5, H, d_k).transpose(1, 2)
scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

mask = torch.tril(torch.ones(5, 5, device=x.device, dtype=torch.bool)).unsqueeze(0).unsqueeze(0)
masked_scores = scores.masked_fill(~mask, float('-inf'))
attn = torch.softmax(masked_scores, dim=-1)

# positions above diagonal must be zero
for i in range(5):
    for j in range(i + 1, 5):
        assert torch.allclose(attn[:, :, i, j], torch.zeros_like(attn[:, :, i, j]), atol=1e-6)

# --- Test 6: Future tokens must not affect past outputs ---
torch.manual_seed(0)
csa3 = CausalSelfAttention(d_model=16, num_heads=2)

x1 = torch.randn(1, 5, 16)
x2 = x1.clone()

# change only the future token at position 4
x2[:, 4, :] += 1000.0

out1 = csa3.forward(x1)
out2 = csa3.forward(x2)

# outputs at positions 0,1,2,3 should be unchanged
assert torch.allclose(out1[:, :4, :], out2[:, :4, :], atol=1e-5)

# output at position 4 may change
assert not torch.allclose(out1[:, 4, :], out2[:, 4, :], atol=1e-5)

print("All causal self-attention tests passed!")


/home/kenneth/CODE/AI_PROJECTS/Leetcode/.venv/lib/python3.10/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


All causal self-attention tests passed!


In [ ]:
import torch
import math

def causal_attention(Q, K, V):
    d_k = K.size(-1)
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)
    S = scores.size(-1)
    mask = torch.triu(torch.ones(S, S, device=scores.device, dtype=torch.bool), diagonal=1)
    scores = scores.masked_fill(mask.unsqueeze(0), float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return torch.bmm(weights, V)

In [ ]:
import torch

# --- Test 1: Output shape ---
out = causal_attention(torch.randn(2, 6, 16), torch.randn(2, 6, 16), torch.randn(2, 6, 16))
assert out.shape == (2, 6, 16)

# --- Test 2: Future tokens don't affect past ---
torch.manual_seed(0)
B, S, D = 1, 8, 16
Q = torch.randn(B, S, D); K = torch.randn(B, S, D); V = torch.randn(B, S, D)
out1 = causal_attention(Q, K, V)
K2, V2 = K.clone(), V.clone()
K2[:, 4:] = torch.randn(B, 4, D); V2[:, 4:] = torch.randn(B, 4, D)
out2 = causal_attention(Q, K2, V2)
assert torch.allclose(out1[:, :4], out2[:, :4], atol=1e-5)

# --- Test 3: First position only sees itself ---
torch.manual_seed(0)
Q = torch.randn(1, 4, 8); K = torch.randn(1, 4, 8); V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
assert torch.allclose(out[:, 0], V[:, 0], atol=1e-5)

# --- Test 4: Gradient flow ---
Q = torch.randn(2, 4, 8, requires_grad=True)
K = torch.randn(2, 4, 8, requires_grad=True)
V = torch.randn(2, 4, 8, requires_grad=True)
causal_attention(Q, K, V).sum().backward()
assert Q.grad is not None and K.grad is not None and V.grad is not None

print('All tests passed!')